In [1]:
%load_ext autoreload
%autoreload 2

# Сравнение MLP, RNN и CNN для классификации изображений

В данной работе проводится сравнительный анализ различных архитектур нейронных сетей.
Цель исследования — определить влияние архитектуры модели на качество классификации изображений.

## Постановка задачи
Задача заключается в классификации медицинских изображений с использованием различных типов нейронных сетей.

В рамках работы рассматриваются три архитектуры:
- Полносвязная нейронная сеть (MLP)
- Рекуррентная нейронная сеть (RNN)
- Сверточная нейронная сеть (CNN)

Все модели обучаются и тестируются на одном и том же датасете, что позволяет провести корректное сравнение их эффективности.

## Датасет

В работе используется датасет Medical MNIST, содержащий медицинские изображения, разделённые на несколько классов.

Перед подачей в модель изображения проходят этап предобработки:
- изменение размера
- преобразование в тензоры
- нормализация

## Загрузка и предобработка данных

На данном этапе выполняется загрузка датасета и применение преобразований к изображениям.

In [2]:
import torch
import torch.nn as nn
import numpy as np
import os
import json
import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns

from models.mlp import MLP
from models.cnn import CNN
from models.rnn import RNN

from utils.train import train
from utils.evaluate import evaluate, count_params, plot_confusion_matrix, get_confusion_matrix
from utils.dataset import get_dataloaders, get_class_names, get_dataset_info

def save_experiment(model, name, acc, losses, time):
    os.makedirs(f"experiments/{name}", exist_ok=True)
    os.makedirs("checkpoints", exist_ok=True)

    torch.save(model.state_dict(), f"checkpoints/{name}.pth")

    results = {
        "model": name,
        "accuracy": acc,
        "train_loss": losses,
        "train_time": time
    }

    with open(f"experiments/{name}/results.json", "w") as f:
        json.dump(results, f, indent=4)

def run_experiment(model, name, train_loader, test_loader, epochs=5, lr=1e-3):
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_losses = []

    start = time.time()

    for epoch in range(epochs):
        loss = train(model, train_loader, optimizer, criterion, device)
        train_losses.append(loss)

        print(f"{name} | Epoch {epoch+1}/{epochs} | Loss: {loss:.4f}")

    end = time.time()
    train_time = end - start
    acc = evaluate(model, test_loader, device)

    save_experiment(model, name, acc, train_losses, train_time)

    return acc, train_losses

device = torch.device("mps" if torch.backends.mps.is_built() else "cpu")
torch.manual_seed(42)

data_dir = "data/Medical_MNIST"
dataset_info = get_dataset_info(data_dir)

print("Характеристики набора данных Medical MNIST:")
dataset_df = pd.DataFrame(dataset_info, index=[0]).T.rename(columns={0: "Value"})
display(dataset_df)

img_size = dataset_info["height"]
input_size = img_size * img_size
gray_scale = (dataset_info["channels"] == 1)

train_loader, test_loader = get_dataloaders(data_dir, batch_size=32, image_size=img_size, gray_scale=gray_scale)
classes = get_class_names(data_dir)

ru_map = {
    "AbdomenCT": "КТ брюшной полости",
    "BreastMRI": "МРТ молочной железы",
    "CXR": "Рентген грудной клетки",
    "ChestCT": "КТ грудной клетки",
    "Hand": "Рентген кисти",
    "HeadCT": "КТ головы"
}

classes_df = pd.DataFrame({
    "Class Name": classes,
    "Class Name (RU)": [ru_map[c] for c in classes]
}).rename(columns={0: "Class Index"})
print("Наименования классов:")
display(classes_df)

Характеристики набора данных Medical MNIST:


,Value
channels,1
height,64
width,64


Наименования классов:


,Class Name,Class Name (RU)
0,AbdomenCT,КТ брюшной полости
1,BreastMRI,МРТ молочной железы
2,CXR,Рентген грудной клетки
3,ChestCT,КТ грудной клетки
4,Hand,Рентген кисти
5,HeadCT,КТ головы


## Эксперимент 1: Полносвязная нейронная сеть (MLP)

Многослойный перцептрон (Multilayer Perceptron, MLP) или полносвязная нейронная сеть — это нейронная сеть, состоящая из последовательности слоёв, где каждый нейрон связан со всеми нейронами предыдущего слоя. MLP является базовой архитектурой глубокого обучения и применяется для решения задач классификации и регрессии. Полносвязная нейронная сеть рассматривает изображение как одномерный вектор признаков.

Пусть входное изображение имеет параметры $(C, H, W)$, где $C$ - количество цветовых каналов, $H$ - высота изображения в пикселях, а $W$ - ширина изображения в пикселях. И пусть оно представимо в виде вектора:

$$x \in \mathbb{R}^n, \quad n = C \times H \times W$$

Данное преобразование осуществляется посредством операции векторизации, при которой многомерный массив размерности $C \times H \times W$ преобразуется в одномерный вектор размерности $CHW$, что приводит к утрате пространственной структуры изображения.
$$\mathbb{R}^{C \times H \times W} \rightarrow \mathbb{R}^{CHW}$$

Полносвязная нейронная сеть реализует отображение:

$$f: \mathbb{R}^n \rightarrow \mathbb{R}^k,$$

где $n$ - размер входа, а $k$ — число классов.

Многослойный перцептрон представляет собой композицию линейных преобразований и нелинейных функций активации:

$x^{(0)} = x$

$x^{(l)} = \sigma\left(W^{(l)} x^{(l-1)} + b^{(l)}\right), \quad l = 1, \dots, L-1$

$z^{(L)} = W^{(L)} x^{(L-1)} + b^{(L)}$

где:

- $x^{(l)}$ — выход $l$-го слоя,
- $W^{(l)} \in \mathbb{R}^{n_l \times n_{l-1}}$ — матрица весов,
- $b^{(l)} \in \mathbb{R}^{n_l}$ — вектор смещений,
- $\sigma(\cdot)$ — нелинейная функция активации,
- $L$ — общее число слоёв сети,
- $z^{(L)}$ - выходной слой.

Выходной слой формирует вектор $z^{(L)} \in \mathbb{R}^k$, который называется вектором логитов.

Логиты представляют собой ненормированные оценки принадлежности объекта к каждому из классов. Они могут принимать произвольные вещественные значения и не ограничены интервалом $[0,1]$. Для получения интерпретируемых вероятностей, т.е выхода модели в задаче многоклассовой классификации применяется функция $\text{softmax}(z^{(L)})$:

$$\hat{y}_i = \frac{e^{z^{(L)}_i}}{\sum_{j=1}^{k} e^{z^{(L)}_j}}, \quad i = 1, \dots, k,$$

 где $\hat{y}$ интерпретируется как вектор вероятностей принадлежности объекта к каждому классу.

Теперь полносвязную нейронную сеть можно рассматривать как параметризованную функцию:

$$f(x; \theta): \mathbb{R}^n \rightarrow \mathbb{R}^k,$$

где $\theta = \{W^{(l)}, b^{(l)}\}_{l=1}^{L}$ — множество обучаемых параметров модели.

Тогда полное преобразование входного вектора $x$ может быть записано в виде композиции:

$$f(x) = f^{(L)} \circ f^{(L-1)} \circ \dots \circ f^{(1)}(x),$$

где каждый слой реализует отображение:

$$f^{(l)}(x) = \sigma(W^{(l)}x + b^{(l)}),$$

а последний слой (классификатор) задаётся линейным преобразованием:

$$f^{(L)}(x) = W^{(L)}x + b^{(L)}.$$

#### Примечание: 
В практической реализации входные данные обрабатываются пакетами (batch), поэтому фактическая размерность входного слоя имеет вид $B \times n$, где $B$ — размер пакета.

### Архитектура используемой модели

В рамках данной работы входными данными являются изображения размерности $1 \times  64 \times 64$, которые предварительно преобразуются в вектор $x \in \mathbb{R}^{4096}$. Рассматриваемая модель представляет собой трёхслойный многослойный перцептрон и задаётся следующими преобразованиями:

$$x^{(1)} = \text{ReLU}\left(W^{(1)} x + b^{(1)}\right), \quad W^{(1)} \in \mathbb{R}^{512 \times 4096},$$

$$x^{(2)} = \text{ReLU}\left(W^{(2)} x^{(1)} + b^{(2)}\right), \quad W^{(2)} \in \mathbb{R}^{256 \times 512},$$

$$z^{(3)} = W^{(3)} x^{(2)} + b^{(3)}, \quad W^{(3)} \in \mathbb{R}^{6 \times 256}.$$

В качестве функции активации используется кусочно-линейная функция ReLU (rectified linear unit):

$$\text{ReLU}(x) = \max(0, x).$$

Реализуемая модель может быть записана как функция:

$$f(x) = W^{(3)} \cdot \text{ReLU}\Big(W^{(2)} \cdot \text{ReLU}(W^{(1)}x + b^{(1)}) + b^{(2)}\Big) + b^{(3)}$$

Выходной слой формирует вектор логитов $z^{(3)} \in \mathbb{R}^6$, который далее используется в функции потерь. Преобразование $\text{softmax}$ в явном виде не применяется, поскольку оно встроено в используемую функцию потерь (кросс-энтропию).

In [ ]:
mlp = MLP(input_size=input_size)
mlp_acc, mlp_loss = run_experiment(mlp, "MLP", train_loader, test_loader)

MLP | Epoch 1/5 | Loss: 0.0890
MLP | Epoch 2/5 | Loss: 0.0341
MLP | Epoch 3/5 | Loss: 0.0210
MLP | Epoch 4/5 | Loss: 0.0298


## Эксперимент 2: Рекуррентная нейронная сеть (RNN)

Для применения рекуррентной сети изображение преобразуется в последовательность (например, по строкам).

Данный подход позволяет использовать временные зависимости, однако не учитывает двумерную структуру изображения в полной мере.


Написать про то, что в реальности для обучения rnn используется алгоритм BPTT и описать его.

In [ ]:
rnn = RNN(input_size=img_size, hidden_size=img_size)
rnn_acc, rnn_loss = run_experiment(rnn, "RNN", train_loader, test_loader)

## Эксперимент 3: Сверточная нейронная сеть (CNN)

Сверточные нейронные сети специально разработаны для обработки изображений.

Они учитывают локальные пространственные зависимости и обладают свойством инвариантности к сдвигу.

In [ ]:
cnn = CNN()
cnn_acc, cnn_loss = run_experiment(cnn, "CNN", train_loader, test_loader)

## Сравнение результатов

### Сводная таблица метрик

В данном разделе проводится количественное сравнение обученных моделей на тестовой выборке.
Для каждой архитектуры (MLP, RNN, CNN) загружаются ранее сохранённые результаты экспериментов, включающие ключевые метрики качества и эффективности.

В частности, для каждой модели анализируются:

* Accuracy — доля правильно классифицированных изображений на тестовой выборке
* Loss — значение функции потерь на последней эпохе обучения
* Time — общее время обучения модели
* Params — количество обучаемых параметров модели

Дополнительно вычисляется количество параметров каждой модели, что позволяет оценить их вычислительную сложность и сравнить эффективность архитектур не только по качеству, но и по ресурсоёмкости.

In [ ]:
def load_results(path):
    with open(path, "r") as f:
        return json.load(f)
    
def load_model(name, device):

    model_map = {
    "MLP": MLP,
    "CNN": CNN,
    "RNN": RNN

}
    model_class = model_map[name]
    
    model = model_class().to(device)
    path = f"checkpoints/{name}.pth"
    
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    
    return model

results_dir = "experiments"
results_list = []

#Временная заглушка, дабы заново не обучать модели при перезапуске ячейки
mlp = load_model("MLP", device)
rnn = load_model("RNN", device)
cnn = load_model("CNN", device)

for model_name in os.listdir(results_dir):
    model_path = os.path.join(results_dir, model_name)

    if os.path.isdir(model_path):
        results_file = os.path.join(model_path, "results.json")

        if os.path.exists(results_file):
            data = load_results(results_file)
            results_list.append(data)

params_dict = {
    "MLP": count_params(mlp),
    "RNN": count_params(rnn),
    "CNN":count_params(cnn)
}

results_df = pd.DataFrame([
    {
        "Model": r["model"],
        "Accuracy": r["accuracy"],
        "Loss": r["train_loss"][-1],
        "Time": r["train_time"]
    }
    for r in results_list
])
results_df["Params"] = results_df["Model"].map(params_dict)

display(results_df)

Наивысшую точность классификации демонстрирует CNN (0.996), что подтверждает её способность эффективно извлекать пространственные признаки изображений. При этом значение функции потерь у данной модели также является минимальным, что указывает на более стабильное и уверенное обучение по сравнению с другими архитектурами.

MLP показывает близкое значение точности (0.994), однако при этом обладает значительно большим числом параметров (более 2 млн). Это указывает на высокую вычислительную сложность модели при отсутствии явных преимуществ в качестве классификации по сравнению с CNN.

RNN демонстрирует наименьшую точность среди трёх моделей (0.986), а также более высокое значение функции потерь. Это связано с тем, что архитектура рекуррентной сети не является естественной для обработки изображений, так как не учитывает двумерную пространственную структуру данных.

### Визуальное сравнение качества моделей

Для наглядного анализа эффективности моделей используются графические методы визуализации. Это позволяет не только сравнить итоговые метрики, но и проанализировать процесс обучения.

Первый график представляет собой столбчатую диаграмму, которая отображает итоговую точность (Accuracy) каждой модели на тестовой выборке.

Каждой архитектуре соответствует свой цвет:

* MLP — красный
* CNN — зелёный
* RNN — синий

Второй график отображает изменение функции потерь (Loss) в процессе обучения по эпохам для каждой модели.

Анализ данного графика позволяет:

- оценить скорость сходимости моделей
- выявить возможное переобучение
- сравнить стабильность обучения различных архитектур


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
baseline = results_df["Accuracy"].min()

models = {
    "MLP": mlp,
    "CNN": cnn,
    "RNN": rnn
}
color_map = {
    "MLP": "Red",
    "CNN": "Green",
    "RNN": "Blue"
}

colors = results_df["Model"].map(color_map)

results_df.plot(
    x="Model",
    y="Accuracy",
    kind="bar",
    legend=False,
    ax=axes[0],
    color=colors
)

axes[0].set_title("Model Comparison (Accuracy)")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(baseline - 0.05, 1.0)

for r in results_list:
    axes[1].plot(r["train_loss"], label=r["model"], color = color_map[r["model"]])

axes[1].set_title("Training Loss Comparison")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

### Анализ ошибок классификации

Для проведения сравнительного анализа характера ошибок каждой модели при одинаковых условиях обучения и тестирования воспользуемся матрицей ошибок (confusion matrix). Данный инструмент позволяет детально исследовать, какие классы изображений модель классифицирует корректно, а какие - путает между собой.

Для каждой архитектуры (MLP, CNN, RNN) выполняется получение предсказаний на тестовой выборке, строится отдельная матрица ошибок и визуализируется в виде тепловой карты, где строки соответствую истинным классам, а столбцы - предсказанным.

Диагональные элементы отражают количество верных классификаций, тогда как внедиагональные элементы показывают ошибки модели.

In [ ]:
metrics_dict = {
    "Model": [],
    "Class": [],
    "Precision": [],
    "Recall": [],
    "F1": []
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, model) in zip(axes, models.items()):
    cm = get_confusion_matrix(model=model,
                              test_loader=test_loader,
                              device=device,)
    plot_confusion_matrix(
        cm=cm,
        classes=classes,
        ax=ax,
        color=color_map[name] + "s",
        title=f"{name} Confusion Matrix"
    )
    recall = cm.diagonal()/cm.sum(axis=1)
    precision = cm.diagonal() / cm.sum(axis=0)
    f1 = 2 * recall * precision / (recall + precision)
    recall = np.nan_to_num(recall)
    precision = np.nan_to_num(precision)
    f1 = np.nan_to_num(f1)

    for i, cls in enumerate(classes):
        metrics_dict["Model"].append(name)
        metrics_dict["Class"].append(cls)
        metrics_dict["Precision"].append(precision[i])
        metrics_dict["Recall"].append(recall[i])
        metrics_dict["F1"].append(f1[i])

plt.tight_layout()
plt.show()

### Оценка качества классификации по отдельным классам

Для более детального анализа качества моделей используются метрики recall, precision и F1-score, которые позволяют оценить поведение модели отдельно для каждого класса изображений.

Recall показывает, какая доля объектов конкретного класса была правильно классифицирована моделью. В отличие от общей точности, данная метрика позволяет выявить способность модели находить все объекты заданного класса.

Precision отражает, насколько корректны предсказания модели для каждого класса, то есть какую долю объектов, отнесённых моделью к данному классу, действительно составляют объекты этого класса.

F1-score является гармоническим средним между precision и recall и используется для комплексной оценки качества классификации.

Метрики вычисляется на основе матрицы ошибок (confusion matrix) следующим образом:

$$Recall_i = \frac{CM_{ii}}{\sum_j CM_{ij}}$$

$$Precision_i = \frac{CM_{ii}}{\sum_j CM_{ji}}$$

$$F1_i = 2 \cdot \frac{Precision_i \cdot Recall_i}{Precision_i + Recall_i}$$

где:

* $CM_{ii}$ — количество правильно классифицированных объектов $i$-го класса
* $\sum_j CM_{ij}$ — общее количество объектов $i$-го класса
* $\sum_j CM_{ji}$ — общее количество объектов, предсказанных как класс i

In [ ]:
metrics_df = pd.DataFrame(metrics_dict)
final_table = metrics_df.pivot_table(
    index=["Class"],
    columns="Model",
    values=["Precision", "Recall", "F1"]
)

display(final_table)

Из полученной таблицы видно, что все модели демонстрируют высокое качество классификации по всем классам, однако между ними наблюдаются различия в характере ошибок и устойчивости.

* CNN показывает наиболее сбалансированные результаты: значения precision, recall и F1-score близки к 1 практически для всех классов. Это говорит о том, что модель не только хорошо находит объекты (высокий recall), но и делает минимальное количество ложных срабатываний (высокий precision). Таким образом, CNN демонстрирует наилучшее обобщение и устойчивость.
* MLP также достигает высоких значений метрик, однако в ряде классов (например, Hand и CXR) наблюдается снижение F1-score, обусловленное дисбалансом между precision и recall. Это связано с тем, что полносвязная сеть не учитывает пространственную структуру изображений, что ухудшает качество разделения схожих классов.
* RNN демонстрирует наибольшее снижение качества, особенно по метрике recall для некоторых классов (например, Hand), что указывает на пропуск части объектов. При этом в отдельных случаях recall остаётся высоким, но снижается precision, что говорит о большем количестве ложных классификаций. Это объясняется тем, что рекуррентная архитектура не предназначена для обработки двумерных изображений.

## Вывод

В ходе работы было проведено сравнение различных архитектур нейронных сетей для задачи классификации изображений.

Полученные результаты показывают:

- Полносвязные сети демонстрируют наихудшие результаты, так как не учитывают структуру изображения.
- Рекуррентные сети показывают среднее качество, поскольку изображение не является естественной последовательностью.
- Сверточные нейронные сети достигают наилучших результатов благодаря учёту пространственных зависимостей.

Таким образом, выбор архитектуры нейронной сети должен соответствовать структуре входных данных.